# Lecture 1

In [6]:
import pandas as pd

# Load dataset
df = pd.read_csv('https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/weatherHistory.csv')
df

In [17]:
df.dtypes

Formatted Date              datetime64[ns, UTC]
Summary                                  object
Precip Type                              object
Temperature (C)                         float64
Apparent Temperature (C)                float64
Humidity                                float64
Wind Speed (km/h)                       float64
Wind Bearing (degrees)                  float64
Visibility (km)                         float64
Loud Cover                              float64
Pressure (millibars)                    float64
Daily Summary                            object
year                                      int32
month                                     int32
day                                       int32
hour                                      int32
dtype: object

## ChatGPT output with no guidance

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# Parse date column into useful features
df["Formatted Date"] = pd.to_datetime(df["Formatted Date"], utc=True)

df["year"] = df["Formatted Date"].dt.year
df["month"] = df["Formatted Date"].dt.month
df["day"] = df["Formatted Date"].dt.day
df["hour"] = df["Formatted Date"].dt.hour

def run_RF(df):
    # Target column
    y = df["Summary"]
    
    # Drop target and columns that may leak text information
    X = df.drop(columns=["Summary", "Formatted Date", "Daily Summary"])
    
    # Identify feature types
    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
    categorical_features = X.select_dtypes(include=["object"]).columns
    
    # Set Up Preprocessing
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])
    
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )
    
    # Set Up Random Forest model
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced_subsample",
            n_jobs=-1
        ))
    ])
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    # Train model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Evaluate
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    # Save trained model
    #joblib.dump(model, "weather_summary_random_forest.pkl")
    #print("\nModel saved as weather_summary_random_forest.pkl")

run_RF(df)

Accuracy: 0.6925282861463059

Classification Report:
                          precision    recall  f1-score   support

Breezy and Mostly Cloudy       0.59      0.59      0.59       103
     Breezy and Overcast       0.72      0.83      0.77       106
Breezy and Partly Cloudy       0.63      0.69      0.66        77
                   Clear       0.77      0.50      0.60      2178
                   Foggy       1.00      1.00      1.00      1430
           Mostly Cloudy       0.63      0.61      0.62      5619
                Overcast       0.74      0.69      0.71      3319
           Partly Cloudy       0.65      0.77      0.70      6347

                accuracy                           0.69     19179
               macro avg       0.72      0.71      0.71     19179
            weighted avg       0.70      0.69      0.69     19179


Confusion Matrix:
[[  61   24   17    0    0    0    1    0]
 [  13   88    2    0    0    1    2    0]
 [  18    4   53    0    0    1    0    1]
 [  

Let's help ChatGPT out a bit. The error comes from some categories that have too few data points. Before we got started, we should have checked what data catigories we had.

In [9]:
counts = df["Summary"].value_counts()
counts

Summary
Partly Cloudy                          31733
Mostly Cloudy                          28094
Overcast                               16597
Clear                                  10890
Foggy                                   7148
Breezy and Overcast                      528
Breezy and Mostly Cloudy                 516
Breezy and Partly Cloudy                 386
Dry and Partly Cloudy                     86
Windy and Partly Cloudy                   67
Light Rain                                63
Breezy                                    54
Windy and Overcast                        45
Humid and Mostly Cloudy                   40
Drizzle                                   39
Breezy and Foggy                          35
Windy and Mostly Cloudy                   35
Dry                                       34
Humid and Partly Cloudy                   17
Dry and Mostly Cloudy                     14
Rain                                      10
Windy                                      8
Hu

There are three catigories with just one entry. Let's remove those and see if the program runs.

In [11]:
df = df[df["Summary"].isin(counts[counts >= 2].index)]
run_RF(df)

Accuracy: 0.5972678450388446

Classification Report:
                          precision    recall  f1-score   support

Breezy and Mostly Cloudy       0.52      0.58      0.55       103
     Breezy and Overcast       0.68      0.78      0.73       106
Breezy and Partly Cloudy       0.61      0.61      0.61        77
                   Clear       0.61      0.32      0.42      2178
                   Foggy       1.00      1.00      1.00      1430
           Mostly Cloudy       0.52      0.51      0.52      5619
                Overcast       0.63      0.58      0.60      3319
           Partly Cloudy       0.56      0.68      0.62      6347

                accuracy                           0.60     19179
               macro avg       0.64      0.63      0.63     19179
            weighted avg       0.60      0.60      0.59     19179


Confusion Matrix:
[[  60   25   18    0    0    0    0    0]
 [  16   83    5    0    0    1    1    0]
 [  26    3   47    0    0    1    0    0]
 [  

It worked! However, ChatGPT would not hold up to a peer review. If I was a reviewer on this model, I would ask the following questions.

1. Are all of these categories necessary and descriptive of our weather system?
2. Are the features the best variables needed to describe the weather?
3. All the features were not included because numeric catigories don't cover all data times.
4. Why were day, month, and year coded into the model directly?
5. Why were hyperparameters not tuned?
6. Why was there no iterative feature selection?
7. Cloud cover was included as a feature, but there was no variance in the data. 